# PolicyProof AI — Tiny Aya Colab Baseline (T4)

**Lexsi 2026 · Experimental inference only.** This notebook verifies your Hugging Face model access, loads the unchanged Tiny Aya model, and benchmarks a fictional policy example. It does **not** fine-tune a model or demonstrate improved safety.

Before running: **Runtime → Change runtime type → T4 GPU**. A read-only Hugging Face token must belong to the account granted access to `CohereLabs/tiny-aya-global`. Never place tokens in cells or GitHub.

## 1. Confirm the T4 GPU

In [ ]:
import torch, subprocess
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Connect to a Colab T4 GPU runtime.")
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run(["nvidia-smi"], check=True)

## 2. Install a compatible Hugging Face stack (fresh runtime only)

The original installation could install huggingface-hub 2.x, which conflicts with the Colab datasets 4.8.5 package. This command pins mutually compatible versions. **Do not reinstall packages while a successfully loaded model is active; apply this on your next fresh runtime, then run the notebook from the start.**

In [ ]:
%pip -q install "transformers==5.22.1" "huggingface-hub>=1.31,<2.0" "datasets==4.8.5" "accelerate>=1.1.0" safetensors sentencepiece
%pip check

## 3. Sign in securely and verify gated access

Create a **read-only** token at https://huggingface.co/settings/tokens. Enter it only through the login popup below.

In [ ]:
from huggingface_hub import notebook_login, whoami, hf_hub_download
notebook_login()
MODEL_ID = "CohereLabs/tiny-aya-global"
hf_hub_download(repo_id=MODEL_ID, filename="config.json")
print("ACCESS CONFIRMED for", whoami()["name"])

## 4. Download PolicyProof and generate the integrity-checked dataset

In [ ]:
import subprocess, pathlib, os, sys
PROJECT = pathlib.Path("/content/PolicyProof-AI")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/betterlifestyle04-commits/PolicyProof-AI.git",
                    str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only"], check=True)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
subprocess.run([sys.executable, "-m", "scripts.prepare"], check=True)
subprocess.run([sys.executable, "-m", "scripts.audit_dataset", "--data-dir", "data"], check=True)
CODE_REVISION = subprocess.check_output(["git","rev-parse","HEAD"], text=True).strip()
print("GitHub commit:", CODE_REVISION)

## 5. Load Tiny Aya on the T4 in float16

A T4 does not support native BF16 compute, so explicitly load this base model as float16. The notebook records the actual Hugging Face repository commit SHA (the previous notebook sometimes displayed `None` for model revision).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True, attn_implementation="sdpa"
)
model.eval()
from huggingface_hub import model_info
MODEL_REVISION = model_info(MODEL_ID).sha
print("Baseline loaded:", MODEL_ID, "revision:", MODEL_REVISION)
print("GPU allocated GiB:", round(torch.cuda.memory_allocated() / 1024**3, 2))

## 6. Generate and score one real model answer

The deterministic benchmark uses **strict full-sentence exact matching**. A concise answer such as `two` can be factually correct yet receive score 0 against the longer gold sentence. Assess factual correctness and citation independently; never claim a 0 means incorrect reasoning.

In [ ]:
from policyproof.core import read_jsonl, make_prompt, score, parse_prediction
cases = read_jsonl(PROJECT / "data/unseen_policy_test.jsonl")
def generate_one(example):
    formatted = tokenizer.apply_chat_template(
        [{"role":"user","content":make_prompt(example)}],
        tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        outputs = model.generate(**formatted, max_new_tokens=160,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
    answer_tokens = outputs[0][formatted["input_ids"].shape[-1]:]
    return tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()
example = cases[0]
prediction = generate_one(example)
print("Example:", example["id"], example["question"])
print("Real model response:", prediction)
print("Valid JSON:", parse_prediction(prediction) is not None)
print("Strict single-case score:", score(example,prediction)["correct"])
print("Reference answer:", example["expected_answer"])
print("Reference source:", example["expected_source_id"])
print("Reminder: strict string comparison is not semantic grading.")

## 7. Optional full 48-case baseline

Now that one genuine response worked, change `RUN_FULL_BASELINE = False` to `True` and run the cell. It loops through 48 questions and records raw predictions, model revision and strict baseline metrics. **Do not confuse the strict score with semantic accuracy.** Download the resulting JSON before the Colab session ends; temporary Colab storage is not persistent.

In [ ]:
RUN_FULL_BASELINE = False
if RUN_FULL_BASELINE:
    import json, datetime
    from policyproof.core import aggregate, sha256
    results = []
    for i, case in enumerate(cases, 1):
        text_out = generate_one(case)
        results.append({"id":case["id"],"raw":text_out,"score":score(case,text_out)})
        if i % 8 == 0: print("Completed",i,"of",len(cases))
    report = {
        "project":"PolicyProof AI / Lexsi 2026",
        "engine":"transformers","is_real_model_result":True,
        "model":MODEL_ID,"model_revision":MODEL_REVISION,
        "model_dtype":"float16","code_commit":CODE_REVISION,
        "is_finetuned":False,"split":"unseen-policy",
        "dataset_sha256":sha256(PROJECT/"data/unseen_policy_test.jsonl"),
        "created_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "metrics":aggregate([r["score"] for r in results]),
        "scores":[r["score"] for r in results],
        "predictions":[{"id":r["id"],"raw":r["raw"]} for r in results],
        "warning":"Actual baseline inference on invented policies; not fine-tuned and not a safety-improvement claim."
    }
    destination = PROJECT/"artifacts"
    destination.mkdir(exist_ok=True)
    path = destination/"tiny_aya_base_unseen_policy.json"
    path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
    print("Saved baseline report:",path)
    print(json.dumps(report["metrics"],indent=2))
else:
    print("Full run disabled; inspect the one-question baseline first.")

## 8. Export results for independent semantic review

After a **full** baseline, run the following cell to download the JSON report and a blind human-review CSV. Reviewers should judge meaning and grounding without seeing the reference answers. A response of `two` backed by the correct source can be counted as semantically correct once verified. **Mock scores are never eligible for review-based model claims.**

In [ ]:
if not RUN_FULL_BASELINE:
    print("Enable and complete the full baseline first; nothing to download.")
else:
    import subprocess, sys
    from google.colab import files
    json_file = PROJECT / "artifacts" / "tiny_aya_base_unseen_policy.json"
    csv_file = PROJECT / "artifacts" / "tiny_aya_blind_review.csv"
    subprocess.run([sys.executable, "-m", "scripts.human_review",
                    "--predictions", str(json_file),
                    "--cases", "data/unseen_policy_test.jsonl",
                    "--output", str(csv_file)], check=True)
    files.download(str(json_file))
    files.download(str(csv_file))

## Next milestone

The baseline is only a starting point. We still need to complete verified CuratorKIT data curation, run an approved AlignTune fine-tuning job, then evaluate the trained model with the unchanged held-out split and AuditKIT. Track semantic review separately from exact match. Don't publish private tokens, and don't claim improved safety based only on synthetic policies.